# 01 — Top-down pipeline only

Questo notebook esegue **solo la prima parte/top-down** della pipeline GORE. La generazione di description, actors, HLG e LLG usa i generatori originali; non viene eseguito il ciclo bottom-up.

Configurazione sperimentale usata qui: **Actors Zero-shot con critic**; **HLG e LLG Few-shot senza critic**. Il flusso resta quello originale del paper: gli LLG vengono generati in un'unica chiamata a partire dall'intero insieme degli HLG. I provider restano quelli nuovi del progetto: Gemini come generator e Qwen/Groq come critic.

Gli output strutturati vengono salvati in `output/top_down/` e possono essere riutilizzati dal notebook bottom-up e dal notebook di valutazione.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("Project root:", PROJECT_ROOT)

from Gemini_API.APIKeysManager import APIKeysManager
from key import get_key_llama, count_Llama_keys

gemini_key = APIKeysManager.get_api_key()
groq_key = get_key_llama()

print("Chiave Gemini presente:", bool(gemini_key))
print("Numero chiavi Gemini:", APIKeysManager.count())
print("Chiave Groq presente:", bool(groq_key))
print("Formato Groq plausibile:", groq_key.startswith("gsk_") if groq_key else False)
print("Numero chiavi Groq:", count_Llama_keys())

## 1. Configurazione

La documentazione segue il percorso originale: se la ground truth contiene `link-readme`, viene chiamato `generate_description(...)`; altrimenti viene usata direttamente la descrizione disponibile nella ground truth. Non viene introdotto chunking aggiuntivo nella baseline.


In [ ]:
import json

from groundtruth import ALL_GROUNDTRUTHS
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
import src.self_critique.refine_response as reflection
from src.examples.shot_learning import ShotPromptingMode
from src.utils import get_markdown
import src.llm_clients as llm_clients

GROUNDTRUTHS = list(ALL_GROUNDTRUTHS)
if not GROUNDTRUTHS:
    raise RuntimeError("No projects found in groundtruth.py")

ACTOR_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
HLG_PROMPTING_MODE = ShotPromptingMode.FEW_SHOT
LLG_PROMPTING_MODE = ShotPromptingMode.FEW_SHOT
ACTOR_EVALUATOR_ABLATION = False
HLG_EVALUATOR_ABLATION = True
LLG_EVALUATOR_ABLATION = True
TOP_DOWN_MAX_ATTEMPTS = 3
TOP_DOWN_QUALITY_THRESHOLD = 8.5
TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = 4
PIPELINE_PROFILE = "ACTORS_ZERO_SHOT_CRITIC_HLG_LLG_FEW_SHOT_NO_CRITIC"
# Keep the established filename so a rerun replaces the stale baseline
# instead of creating duplicate datasets for the bottom-up notebook.
OUTPUT_FILE_PROFILE = "ZERO_SHOT_feedback_original_top_down"

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_DIR.mkdir(parents=True, exist_ok=True)

def top_down_output_file_for(dataset_name: str) -> Path:
    return TOP_DOWN_DIR / f"{dataset_name}_{OUTPUT_FILE_PROFILE}_top_down.json"

def run_top_down_reflection(**kwargs):
    """Run the paper loop parameters without changing bottom-up defaults."""
    previous_threshold = reflection.QUALITY_THRESHOLD
    previous_example_count = reflection.EVALUATOR_FEW_SHOT_EXAMPLE_COUNT
    reflection.QUALITY_THRESHOLD = TOP_DOWN_QUALITY_THRESHOLD
    reflection.EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = (
        TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT
    )
    try:
        return generate_response_with_reflection(
            max_attempts=TOP_DOWN_MAX_ATTEMPTS,
            **kwargs,
        )
    finally:
        reflection.QUALITY_THRESHOLD = previous_threshold
        reflection.EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = previous_example_count

print(f"Ground-truth projects discovered: {len(GROUNDTRUTHS)}")
for project_index, project in enumerate(GROUNDTRUTHS, start=1):
    print(f"  {project_index}/{len(GROUNDTRUTHS)} - {project['name']}")
print("Top-down output directory:", TOP_DOWN_DIR)
print("Gemini generator:", llm_clients.GEMINI_MODEL)
print("Preferred Groq evaluator:", llm_clients.PREFERRED_GROQ_EVALUATOR_MODEL)
print("Groq fallback evaluator:", llm_clients.GROQ_EVALUATOR_FALLBACK_MODEL)
print("Groq evaluator max output tokens:", llm_clients.GROQ_LLAMA_MAX_TOKENS)
print("Actor prompting:", ACTOR_PROMPTING_MODE.name, "| critic: ON")
print("HLG prompting:", HLG_PROMPTING_MODE.name, "| critic: OFF")
print("LLG prompting:", LLG_PROMPTING_MODE.name, "| critic: OFF")
print("Top-down evaluator few-shot examples:", TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT)
print("Top-down quality threshold:", TOP_DOWN_QUALITY_THRESHOLD)
print("Top-down max reflection attempts:", TOP_DOWN_MAX_ATTEMPTS)


## 2. Esecuzione top-down e salvataggio

Per ogni dataset viene costruita la `description` con il percorso originale. Gli Actors vengono generati in Zero-shot e raffinati dal critic Few-shot. Gli HLG e gli LLG vengono generati in Few-shot senza critic. Come nella pipeline originale, tutti gli HLG vengono passati insieme all'unica chiamata che genera gli LLG. Il JSON conserva sia le liste testuali sia gli oggetti Pydantic strutturati necessari al confronto sperimentale e al successivo bottom-up.


In [ ]:
generated_descriptions = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}


def run_top_down_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]

    try:
        # ---------------------------------------------------------------
        # 1. ORIGINAL documentation preprocessing
        # ---------------------------------------------------------------
        if "link-readme" in groundtruth:
            documentation_link = groundtruth["link-readme"]

            # Saved only for traceability; generate_description() below follows
            # the original repository path and reads the README itself.
            raw_description = get_markdown(documentation_link)
            generated_description = generate_description(documentation_link)
            description = generated_description.description

            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "original_generate_description",
            }
        else:
            raw_description = str(groundtruth["description"])
            description = raw_description

            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "groundtruth_description",
            }

        print(
            f"[{dataset_name}] Documentation ready | "
            f"raw={len(raw_description)} chars | "
            f"pipeline={len(description)} chars"
        )

        # ---------------------------------------------------------------
        # 2. ACTORS: Zero-shot + critic/reflection
        # ---------------------------------------------------------------
        actors, actors_score, actors_critique = run_top_down_reflection(
            target_type="Actors",
            call_function=generate_actors,
            define_args=(description,),
            eval_mode=EvalMode.ACTORS,
            eval_args=(description,),
            shotPromptingMode=ACTOR_PROMPTING_MODE,
            llama_ablation=ACTOR_EVALUATOR_ABLATION,
        )

        # ---------------------------------------------------------------
        # 3. HLG: Few-shot, critic ablated
        # ---------------------------------------------------------------
        high_level_goals, hl_score, hl_critique = (
            run_top_down_reflection(
                target_type="High Level Goals",
                call_function=generate_high_level_goals,
                define_args=(description, actors),
                eval_mode=EvalMode.HIGH_LEVEL,
                eval_args=(description, actors),
                shotPromptingMode=HLG_PROMPTING_MODE,
                llama_ablation=HLG_EVALUATOR_ABLATION,
            )
        )

        # ---------------------------------------------------------------
        # 4. LLG: all HLGs in one call, Few-shot, critic ablated
        # ---------------------------------------------------------------
        low_level_goals, ll_score, ll_critique = run_top_down_reflection(
            target_type="Low Level Goals",
            call_function=generate_low_level_goals,
            define_args=(high_level_goals,),
            eval_mode=EvalMode.LOW_LEVEL,
            eval_args=(description, actors, high_level_goals),
            shotPromptingMode=LLG_PROMPTING_MODE,
            llama_ablation=LLG_EVALUATOR_ABLATION,
        )

        generated_descriptions[dataset_name] = description
        generated_actor_objects[dataset_name] = actors
        generated_hl_objects[dataset_name] = high_level_goals
        generated_ll_objects[dataset_name] = low_level_goals

        output = {
            "name": dataset_name,
            "description": description,
            "rawDescription": raw_description,
            "documentationPreprocessing": preprocessing_info,
            "actors": [actor.name for actor in actors.actors],
            "highLevelGoals": [
                goal.description for goal in high_level_goals.goals
            ],
            "lowLevelGoals": [
                goal.description for goal in low_level_goals.low_level_goals
            ],
            "structuredActors": actors.model_dump(mode="json"),
            "structuredHighLevelGoals": high_level_goals.model_dump(mode="json"),
            "structuredLowLevelGoals": low_level_goals.model_dump(mode="json"),
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
            "baselineConfiguration": {
                "generatorModel": llm_clients.GEMINI_MODEL,
                "actorPromptingMode": ACTOR_PROMPTING_MODE.name,
                "actorEvaluatorEnabled": True,
                "highLevelGoalPromptingMode": HLG_PROMPTING_MODE.name,
                "lowLevelGoalPromptingMode": LLG_PROMPTING_MODE.name,
                "highLevelGoalEvaluatorEnabled": False,
                "lowLevelGoalEvaluatorEnabled": False,
                "goalEvaluatorModel": llm_clients.GROQ_EVALUATOR_MODEL,
                "qualityThreshold": TOP_DOWN_QUALITY_THRESHOLD,
                "maxReflectionAttempts": TOP_DOWN_MAX_ATTEMPTS,
                "evaluatorFewShotExampleCount": TOP_DOWN_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT,
            },
            "pipelineConfiguration": {
                "actors": {
                    "promptingMode": ACTOR_PROMPTING_MODE.name,
                    "evaluator": True,
                },
                "highLevelGoals": {
                    "promptingMode": HLG_PROMPTING_MODE.name,
                    "evaluator": False,
                },
                "lowLevelGoals": {
                    "promptingMode": LLG_PROMPTING_MODE.name,
                    "evaluator": False,
                    "generationStrategy": "all_hlgs_in_one_call",
                },
                "profile": PIPELINE_PROFILE,
            },
        }

        output_file = top_down_output_file_for(dataset_name)
        with output_file.open("w", encoding="utf-8") as file:
            json.dump(
                output,
                file,
                indent=4,
                ensure_ascii=False,
                default=str,
            )

        print(
            f"[{dataset_name}] saved: {output_file.name} | "
            f"Actors={len(actors.actors)} (Zero-shot, evaluator ON) | "
            f"HLG={len(high_level_goals.goals)} (Few-shot, evaluator OFF) | "
            f"LLG={len(low_level_goals.low_level_goals)} "
            f"(Few-shot, evaluator OFF, single global call) | "
            f"evaluator={llm_clients.GROQ_EVALUATOR_MODEL}"
        )

    except Exception as error:
        baseline_errors[dataset_name] = (
            f"{type(error).__name__}: {error}"
        )
        print(
            f"[{dataset_name}] Top-down extraction failed: "
            f"{type(error).__name__}: {error}"
        )

        import traceback
        traceback.print_exc()


for project_index, groundtruth in enumerate(GROUNDTRUTHS, start=1):
    print(
        f"\n=== PROJECT {project_index}/{len(GROUNDTRUTHS)}: "
        f"{groundtruth['name']} ==="
    )
    run_top_down_for_dataset(groundtruth)


## 3. Riepilogo

Al termine viene mostrato il numero di dataset completati e falliti. Ogni dataset completato produce un JSON separato in `output/top_down/`.


In [ ]:
completed_outputs = globals().get("generated_ll_objects", {})
execution_errors = globals().get("baseline_errors", {})
output_directory = globals().get("TOP_DOWN_DIR")

print("\nTop-down execution summary")
print("--------------------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Completed datasets: {len(completed_outputs)}")
print(f"Failed datasets: {len(execution_errors)}")
print(f"Output directory: {output_directory or 'pipeline not executed yet'}")

if execution_errors:
    print("Errors:")
    for dataset_name, error in execution_errors.items():
        print(f"- {dataset_name}: {error}")
elif not completed_outputs:
    print("Pipeline not executed: run all previous cells first.")